<a href="https://colab.research.google.com/github/dougyd92/ML-Foudations/blob/main/Projects/Project_4_Handwriting_Neural_Net__pytorch_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project: Neural Network Classification of Handwritten Letters

## Overview

In this project, you will design, implement, and train a neural network to classify handwritten letters using the EMNIST Letters dataset. This dataset contains 28×28 grayscale images of handwritten letters (A–Z), providing a more challenging classification task than the classic MNIST digit dataset.

Your classifier will be evaluated on a separate, held-out test set that you will not have access to during development. This test set consists of handwritten letters created independently from the EMNIST dataset, so your model must generalize well to truly unseen data.

## Learning Objectives

By completing this project, you will:

- Gain hands-on experience loading and preprocessing image data for neural network training
- Design and implement neural network architectures using PyTorch
- Apply techniques such as data augmentation, regularization, and hyperparameter tuning
- Evaluate model performance and iterate on your design
- Develop intuition for the tradeoffs between model complexity, training time, and generalization

## The Competition

This project is a competition. The student whose model achieves the **highest accuracy on the held-out test set wins** and will receive a shoutout on LinkedIn.

May the best model win!

## Submission Requirements

1. This completed Jupyter notebook with all code cells executed
2. Your exported model saved as `submission_model.pt` (generated by the submission cell at the end)

That's it: **two files.** The submission cell handles model export automatically. No separate `predict.py` needed.

## Important Notes

- **EMNIST Orientation**: EMNIST images are stored transposed and flipped. You must correct this when loading the data.
- **Label Indexing**: EMNIST Letters uses 1-indexed labels (1–26 for A–Z). You may want to convert to 0-indexed for compatibility with PyTorch's cross-entropy loss.
- **No Pretrained Models**: You must train your model from scratch. Do not use pretrained weights or transfer learning.
- **Parameter Limit**: Your model must have **1 million parameters or fewer**. Models exceeding this limit will be disqualified. Code is provided below to check your parameter count.
- **Reproducibility**: Set random seeds for reproducibility and document any non-deterministic choices.
- **Hardware**: If running in Colab, be sure to go to "Change runtime type" and select either a GPU or TPU instead of the default CPU. If running on your own machine, be sure you have the correct drivers to take advantage of your GPU.

---

## Section 1: Setup and Imports

Import all necessary libraries for the project. You will need at minimum:

- PyTorch and torchvision for model building and data loading
- NumPy for numerical operations
- Matplotlib for visualization

You may also find these useful:
- `torch.optim` for optimizers
- `torch.nn.functional` for activation functions and loss computation
- `torchvision.transforms` for data preprocessing and augmentation
- `sklearn.metrics` for additional evaluation metrics

Set your random seeds here for reproducibility.

In [ ]:
# Your imports here


In [ ]:
# Set random seeds for reproducibility
# Consider: torch.manual_seed(), numpy random seed, and CUDA seed if using GPU


In [ ]:
# Check for GPU availability and set device


---

## Section 2: Data Loading and Exploration

Load the EMNIST Letters dataset using torchvision. The dataset will be automatically downloaded if not present.

### Tasks:

1. Load the training and test splits of EMNIST Letters
2. Explore the dataset: How many samples? What are the image dimensions? How many classes?
3. Visualize several example images from different classes
4. Check the class distribution—is the dataset balanced?

### Important: Fixing EMNIST Orientation

EMNIST images are stored transposed and flipped compared to how they should appear. You need to apply a transformation to correct this:

```python
# The image needs to be transposed and flipped to appear correctly
# This can be done with a lambda transform or by defining a custom transform
```

Verify your orientation fix by visualizing some images—letters should be readable.

In [ ]:
# Load EMNIST Letters dataset
from torchvision import datasets, transforms
import numpy as np

# Custom transform to fix EMNIST orientation (images are transposed and flipped)
class FixEMNISTOrientation:
    """EMNIST images are stored transposed and flipped. This corrects them."""
    def __call__(self, img):
        return transforms.functional.rotate(img, -90).transpose(Image.FLIP_LEFT_RIGHT)

# Basic transform for initial exploration (just fix orientation and convert to tensor)
from PIL import Image
explore_transform = transforms.Compose([
    FixEMNISTOrientation(),
    transforms.ToTensor()
])

# Load training and test sets
train_dataset = datasets.EMNIST(
    root='./data',
    split='letters',
    train=True,
    download=True,
    transform=explore_transform
)

test_dataset = datasets.EMNIST(
    root='./data',
    split='letters',
    train=False,
    download=True,
    transform=explore_transform
)

# Note: EMNIST Letters labels are 1-indexed (1-26 for A-Z)
# You may want to subtract 1 to make them 0-indexed for PyTorch CrossEntropyLoss
print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Image shape: {train_dataset[0][0].shape}")
print(f"Label range: {min(train_dataset.targets)} to {max(train_dataset.targets)}")

In [ ]:
# The dataset is already loaded above. Add any additional exploration here.
# For example, you might want to check unique classes, compute statistics, etc.


In [ ]:
# Visualize sample images
import matplotlib.pyplot as plt

def label_to_letter(label):
    """Convert 1-indexed label to letter. Label 1 = 'A', 2 = 'B', etc."""
    return chr(label - 1 + ord('A'))

fig, axes = plt.subplots(2, 8, figsize=(12, 4))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f'{label_to_letter(label)}')
    ax.axis('off')
plt.suptitle('Sample EMNIST Letters (orientation corrected)')
plt.tight_layout()
plt.show()

In [ ]:
# Analyze class distribution
# Are all letters equally represented in the training set?


### Notes

Use this space to jot down observations about the dataset—anything that might inform your modeling decisions. For example: Are some letters likely to be confused with each other? Is the dataset balanced?

*Your notes:*



---

## Section 3: Data Preprocessing and Augmentation

Prepare your data for training by defining appropriate transformations. This section is critical for achieving good generalization.

### Required Preprocessing:

- Fix the EMNIST transpose/flip issue
- Convert images to tensors
- Normalize pixel values (consider using EMNIST statistics or standard [0,1] normalization)
- Adjust labels from 1-indexed to 0-indexed

### Suggested Data Augmentation (choose based on your experimentation):

- **Random rotation**: Small rotations (±10-15°) simulate natural writing variation
- **Random affine transforms**: Small translations, scaling, or shearing
- **Elastic deformation**: Simulates the natural variability in handwriting
- **Random erasing/cutout**: Can improve robustness
- **Gaussian noise**: Adds robustness to noise in input

**Important**: Data augmentation should typically only be applied to training data, not validation or test data.

### Creating a Validation Split:

You should create a validation set from the training data to monitor for overfitting and tune hyperparameters. A typical split is 80-90% training, 10-20% validation.

### ⚠️ Track Your Normalization Values

If you use `transforms.Normalize(mean, std)` in your preprocessing pipeline, **write down the exact mean and std values you used.** You will need them in the submission cell at the end of the notebook. The submission cell bakes your normalization into the exported model so that grading works consistently across all submissions.

If you only use `transforms.ToTensor()` with no `Normalize()`, you don't need to do anything extra.

In [ ]:
# Define transforms for training data
# Include orientation fix, normalization, and any augmentation you choose


In [ ]:
# Define transforms for validation/test data
# Should include orientation fix and normalization, but typically NO augmentation


In [ ]:
# Reload datasets with transforms applied


In [ ]:
# Create train/validation split
# Hint: Use torch.utils.data.random_split or sklearn's train_test_split


In [ ]:
# Create DataLoaders for train, validation, and test sets
# Consider appropriate batch sizes (32, 64, 128 are common choices)


In [ ]:
# Visualize some augmented training samples to verify your transforms look reasonable


### Preprocessing Notes

Document any preprocessing decisions worth remembering—normalization strategy, augmentation choices, batch size, etc.

*Your notes:*



---

## Section 4: Model Architecture

Design and implement your neural network architecture. You have flexibility in your design choices, but your model must be trained from scratch (no pretrained weights).


### ⚠️ Parameter Limit: 1,000,000

Your model must have **1 million trainable parameters or fewer**. Models exceeding this limit will be disqualified. Use the parameter counting code provided below to verify your model is within the limit.

### Architecture Options to Consider:

**Fully Connected Networks:**
- Simpler to implement
- May require more parameters to achieve good performance
- Don't inherently capture spatial structure

**Convolutional Neural Networks (CNNs):**
- Better suited for image data
- Capture local spatial patterns
- Typically more parameter-efficient for image tasks

### Design Considerations:

- **Depth**: How many layers? Deeper networks can learn more complex features but are harder to train
- **Width**: How many neurons/filters per layer?
- **Activation functions**: ReLU is standard, but consider LeakyReLU, ELU, or GELU
- **Regularization**: Dropout, batch normalization, weight decay
- **Pooling**: Max pooling, average pooling, or strided convolutions

### Suggested Starting Points:

For a CNN, a reasonable starting architecture might be:
- 2-3 convolutional blocks (conv → activation → pooling)
- 1-2 fully connected layers
- Dropout for regularization
- Output layer with 26 neurons (one per letter)

Start simple and add complexity as needed!

In [ ]:
# Define your neural network architecture

class LetterClassifier(nn.Module):
    def __init__(self):
        super(LetterClassifier, self).__init__()
        # Define your layers here
        pass

    def forward(self, x):
        # Define forward pass
        pass


In [ ]:
# Instantiate your model and move to device (GPU if available)


In [ ]:
# Print model summary and check parameter limit
PARAMETER_LIMIT = 1_000_000

def count_parameters(model):
    """Count trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

num_params = count_parameters(model)
print(f"Model Architecture:")
print(model)
print(f"\n" + "="*50)
print(f"Total trainable parameters: {num_params:,}")
print(f"Parameter limit:            {PARAMETER_LIMIT:,}")
print(f"="*50)

if num_params > PARAMETER_LIMIT:
    print(f"\n⚠️  WARNING: Your model has {num_params - PARAMETER_LIMIT:,} parameters over the limit!")
    print(f"    You must reduce your model size to be eligible for the competition.")
else:
    print(f"\n✓ Your model is within the parameter limit.")
    print(f"  Remaining budget: {PARAMETER_LIMIT - num_params:,} parameters")


In [ ]:
# Print model summary
# Show the architecture and count the total number of trainable parameters


In [ ]:
# Verify the model works by passing a sample batch through it


### Architecture Notes

Document your architectural choices—why this type of network, how you chose depth/width, regularization techniques, parameter count, etc.

*Your notes:*



---

## Section 5: Training Setup

Configure the training process by selecting a loss function, optimizer, and learning rate schedule.

### Loss Function:

For multi-class classification, `CrossEntropyLoss` is the standard choice. Note that PyTorch's CrossEntropyLoss combines LogSoftmax and NLLLoss, so your model's forward pass should output raw logits (no softmax).

### Optimizer Options:

- **SGD**: Classic choice, may require careful learning rate tuning and momentum
- **Adam**: Adaptive learning rates, often works well out of the box
- **AdamW**: Adam with decoupled weight decay, often better for regularization

### Learning Rate Schedule (optional but recommended):

- **StepLR**: Reduce LR by a factor every N epochs
- **ReduceLROnPlateau**: Reduce LR when validation loss plateaus
- **CosineAnnealingLR**: Smoothly decrease LR following a cosine curve
- **OneCycleLR**: Increases then decreases LR, can achieve faster convergence

### Hyperparameters to Tune:

- Learning rate (try values like 0.001, 0.01, 0.0001)
- Weight decay (L2 regularization)
- Number of epochs
- Optimizer-specific parameters (momentum, betas for Adam)

In [ ]:
# Define loss function


In [ ]:
# Define optimizer


In [ ]:
# (Optional) Define learning rate scheduler


In [ ]:
# Set training hyperparameters
num_epochs = None  # Choose an appropriate number


---

## Section 6: Training Loop

Implement the training loop. Your training loop should:

1. Iterate over epochs
2. For each epoch, iterate over batches in the training set
3. Perform forward pass, compute loss, backward pass, and optimizer step
4. Track training loss and accuracy
5. Evaluate on validation set after each epoch
6. Track validation loss and accuracy
7. Implement early stopping or model checkpointing (save the best model)

### Best Practices:

- Set model to training mode (`model.train()`) during training
- Set model to evaluation mode (`model.eval()`) during validation
- Use `torch.no_grad()` during validation to save memory
- Save the model with the best validation accuracy
- Print progress regularly so you can monitor training

In [ ]:
# Implement training function for one epoch

def train_one_epoch(model, train_loader, criterion, optimizer, device):
    """Train the model for one epoch.

    Returns:
        tuple: (average_loss, accuracy)
    """
    pass


In [ ]:
# Implement validation/evaluation function

def evaluate(model, data_loader, criterion, device):
    """Evaluate the model on a dataset.

    Returns:
        tuple: (average_loss, accuracy)
    """
    pass


In [ ]:
# Main training loop
# Track training history for plotting
# Save the best model based on validation accuracy

train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []
best_val_accuracy = 0.0

# Your training loop here


---

## Section 7: Training Visualization and Analysis

Visualize your training progress and analyze the results.

### Required Plots:

1. Training and validation loss over epochs
2. Training and validation accuracy over epochs

### Analysis Questions:

- Does the model show signs of overfitting? How can you tell?
- Did the learning rate schedule (if used) have a visible effect?
- At which epoch did the model achieve its best validation performance?

In [ ]:
# Plot training and validation loss


In [ ]:
# Plot training and validation accuracy


### Training Notes

Observations on your training curves—signs of overfitting/underfitting, best validation accuracy, effects of learning rate schedule, etc.

*Your notes:*



---

## Section 8: Model Evaluation

Load your best model and perform a thorough evaluation on the EMNIST test set.

### Required Analysis:

1. Report overall test accuracy
2. Generate a confusion matrix
3. Identify which letter pairs are most commonly confused
4. Visualize some correctly classified and misclassified examples

In [ ]:
# Load the best model


In [ ]:
# Evaluate on EMNIST test set


In [ ]:
# Generate predictions for confusion matrix


In [ ]:
# Plot confusion matrix
# Hint: Use sklearn.metrics.confusion_matrix and display with matplotlib or seaborn


In [ ]:
# Identify the most confused letter pairs
# Which letters does your model most frequently mix up?


In [ ]:
# Visualize correctly classified examples


In [ ]:
# Visualize misclassified examples
# Show the image, predicted label, and true label


### Evaluation Notes

Observations from your evaluation—test accuracy, which letters are confused, patterns in misclassifications, etc.

*Your notes:*



---

## Section 9: Experimentation Log

Document the experiments you ran while developing your model. Tracking your experiments helps you understand what works and why.

For each significant experiment, record:
- What you changed
- Your hypothesis for why it might help
- The parameter count
- The result (validation accuracy)
- What you learned

### Example Format:

| Experiment | Change | Hypothesis | Val Accuracy | Notes |
|------------|--------|------------|--------------|-------|
| Baseline | Initial CNN | - | 85.2% | Starting point |
| Exp 1 | Added dropout (0.5) | Reduce overfitting | 87.1% | Clear improvement |
| Exp 2 | Increased filters (32→64) | More capacity | 86.8% | No improvement, reverted |

### Your Experimentation Log

| Experiment | Change | Hypothesis | Val Accuracy | Notes |
|------------|--------|------------|--------------|-------|
| | | | | |
| | | | | |
| | | | | |
| | | | | |
| | | | | |

### Reflection

What worked? What surprised you? What would you try with more time?

*Your notes:*



---

## Section 10: Model Export and Submission

Run the cells below to export your model for grading. The export process wraps your model and its normalization into a single file so that all submissions are evaluated consistently.

### How it works:

1. You enter the normalization values you used during training (if any)
2. The submission cell wraps your model in a thin layer that applies normalization internally
3. The wrapped model is exported using `torch.jit.trace`, which saves both the architecture and weights into one file
4. Your grading script feeds raw [0, 1] pixel values; the exported model handles the rest

**This means you don't need to write a separate predict.py or worry about preprocessing mismatches.**

### Step 1: Enter your normalization values

If you used `transforms.Normalize(mean, std)` in your training/validation transforms, enter the same values below. If you did not use `Normalize()` (i.e., you only used `ToTensor()`), leave the defaults as `0.0` and `1.0`.

In [ ]:
# ============================================================
# FILL IN YOUR NORMALIZATION VALUES
# ============================================================
# If you used: transforms.Normalize((0.1722,), (0.3309,))
# Then set:    NORM_MEAN = 0.1722
#              NORM_STD  = 0.3309
#
# If you only used ToTensor() with no Normalize(), leave these as-is.
NORM_MEAN = 0.0
NORM_STD = 1.0

### Step 2: Export your model

**Run this cell without modifications.** It wraps your model with the normalization values above, exports it as a traced model, and verifies the export is correct.

In [ ]:
#@title === SUBMISSION EXPORT (DO NOT MODIFY THIS CELL) ===

import torch
import torch.nn as nn

# --- Submission wrapper: bakes normalization into the model ---
class _SubmissionWrapper(nn.Module):
    def __init__(self, model, mean, std):
        super().__init__()
        self.model = model
        use_norm = not (mean == 0.0 and std == 1.0)
        self.use_norm = use_norm
        if use_norm:
            self.register_buffer('mean', torch.tensor(mean).float().view(1, 1, 1, 1))
            self.register_buffer('std', torch.tensor(std).float().view(1, 1, 1, 1))

    def forward(self, x):
        if self.use_norm:
            x = (x - self.mean) / self.std
        return self.model(x)

# --- Parameter check ---
PARAMETER_LIMIT = 1_000_000
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {num_params:,}")
if num_params > PARAMETER_LIMIT:
    raise ValueError(
        f"❌ Model has {num_params:,} parameters, exceeding the "
        f"{PARAMETER_LIMIT:,} limit. Reduce your model size before submitting."
    )
print(f"✓ Parameter check passed ({num_params:,} ≤ {PARAMETER_LIMIT:,})")

# --- Wrap and trace ---
model.eval()
model.cpu()

wrapped = _SubmissionWrapper(model, NORM_MEAN, NORM_STD)
wrapped.eval()

dummy = torch.randn(1, 1, 28, 28)
with torch.no_grad():
    expected_output = wrapped(dummy)

assert expected_output.shape[1] == 26, (
    f"❌ Expected 26 output classes, got {expected_output.shape[1]}. "
    f"Make sure your output layer has 26 neurons (one per letter, 0-indexed A-Z)."
)

traced = torch.jit.trace(wrapped, dummy)
torch.jit.save(traced, 'submission_model.pt')

# --- Verify the saved model loads and matches ---
loaded = torch.jit.load('submission_model.pt')
loaded.eval()
with torch.no_grad():
    reload_output = loaded(dummy)

assert torch.allclose(expected_output, reload_output, atol=1e-5), (
    "❌ Reload verification failed: outputs don't match. "
    "This shouldn't happen. Please reach out for help."
)

print(f"✓ Output shape: {reload_output.shape} (batch_size, 26)")
norm_str = f"Normalize(mean={NORM_MEAN}, std={NORM_STD})" if not (NORM_MEAN == 0.0 and NORM_STD == 1.0) else "None (raw [0,1] input)"
print(f"✓ Normalization baked in: {norm_str}")
print(f"✓ Model exported to: submission_model.pt")
print()
print("This file is ready to submit. It contains your full model,")
print("weights, and normalization, all in one file.")

### Step 3: Sanity check your exported model

This cell loads your exported model and tests it on sample images. It runs in two modes:

**Mode 1 (default): EMNIST test set.** Works immediately with no setup. Uses EMNIST test images with raw [0, 1] pixels (no `Normalize`, no augmentation) to confirm the exported model produces reasonable predictions. This simulates what the grading script does.

**Mode 2: Sample images folder.** After uploading the sample images folder to Colab, set `USE_SAMPLE_IMAGES = True` and update `SAMPLE_FOLDER` to point to the uploaded folder. This tests your model on the same kind of independently-created handwritten images that will be used for final grading.

⚠️ **The sample images do NOT need the EMNIST orientation fix.** They are already correctly oriented. The preprocessing for sample images is just: convert to grayscale → scale to [0, 1]. The cell below handles this automatically.

In [ ]:
# ============================================================
# SANITY CHECK CONFIGURATION
# ============================================================
# Mode 1 (default): test on EMNIST data
# Mode 2: set USE_SAMPLE_IMAGES = True and point to your uploaded folder
USE_SAMPLE_IMAGES = False
SAMPLE_FOLDER = '/content/sample_images'  # Update this path if needed

In [ ]:
import os
import glob
from PIL import Image

loaded_model = torch.jit.load('submission_model.pt')
loaded_model.eval()

if USE_SAMPLE_IMAGES:
    # ----- Mode 2: Sample images from uploaded folder -----
    image_paths = sorted(glob.glob(os.path.join(SAMPLE_FOLDER, '*.png')))
    if not image_paths:
        image_paths = sorted(glob.glob(os.path.join(SAMPLE_FOLDER, '*.jpg')))
    assert len(image_paths) > 0, (
        f"No images found in {SAMPLE_FOLDER}. "
        f"Make sure you uploaded the folder and the path is correct."
    )

    images, true_letters, pred_letters = [], [], []
    for path in image_paths:
        # Extract ground-truth label from filename (first character)
        true_letter = os.path.basename(path)[0].upper()
        true_letters.append(true_letter)

        # Preprocessing: grayscale, resize to 28x28 (if needed), scale to [0,1]
        # NO EMNIST orientation fix — these images are already correctly oriented
        img = Image.open(path).convert('L')
        if img.size != (28, 28):
            img = img.resize((28, 28), Image.BILINEAR)
        img_tensor = transforms.ToTensor()(img)  # [1, 28, 28], values in [0, 1]
        images.append(img_tensor)

        with torch.no_grad():
            logits = loaded_model(img_tensor.unsqueeze(0))
            pred = logits.argmax(dim=1).item()
        pred_letters.append(chr(pred + ord('A')))

    # Show results
    n_show = min(len(images), 16)
    n_cols = min(n_show, 8)
    n_rows = (n_show + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2 * n_cols, 2.8 * n_rows))
    if n_rows == 1:
        axes = [axes] if n_cols == 1 else list(axes)
    else:
        axes = [ax for row in axes for ax in row]

    for i in range(n_show):
        ax = axes[i]
        ax.imshow(images[i].squeeze(), cmap='gray')
        color = 'green' if pred_letters[i] == true_letters[i] else 'red'
        ax.set_title(f"True: {true_letters[i]}
Pred: {pred_letters[i]}", color=color, fontsize=10)
        ax.axis('off')
    for i in range(n_show, len(axes)):
        axes[i].axis('off')

    correct = sum(p == t for p, t in zip(pred_letters, true_letters))
    title = f"Sample Images: {correct}/{len(true_letters)} correct ({100*correct/len(true_letters):.0f}%)"
    plt.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()

    # Print per-image results
    print(f"\nResults on {len(true_letters)} sample images:")
    for i, (path, true, pred) in enumerate(zip(image_paths, true_letters, pred_letters)):
        status = "✓" if true == pred else "✗"
        print(f"  {status} {os.path.basename(path):20s}  true={true}  pred={pred}")
    print(f"\nAccuracy: {correct}/{len(true_letters)} ({100*correct/len(true_letters):.1f}%)")

else:
    # ----- Mode 1 (default): EMNIST test set -----
    sanity_transform = transforms.Compose([
        FixEMNISTOrientation(),
        transforms.ToTensor()
    ])
    sanity_dataset = datasets.EMNIST(
        root='./data', split='letters', train=False,
        download=False, transform=sanity_transform
    )

    fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
    indices = np.random.choice(len(sanity_dataset), 8, replace=False)

    for ax, idx in zip(axes, indices):
        img, label = sanity_dataset[idx]
        true_letter = chr((label - 1) + ord('A'))

        with torch.no_grad():
            logits = loaded_model(img.unsqueeze(0))
            pred = logits.argmax(dim=1).item()
        pred_letter = chr(pred + ord('A'))

        color = 'green' if pred_letter == true_letter else 'red'
        ax.imshow(img.squeeze(), cmap='gray')
        ax.set_title(f"True: {true_letter}\nPred: {pred_letter}", color=color, fontsize=10)
        ax.axis('off')

    plt.suptitle("Sanity Check: Exported Model on EMNIST Test Images", fontsize=12)
    plt.tight_layout()
    plt.show()

---

## Section 11: Final Summary

Provide a brief summary of your project.

### Project Summary

**Model Architecture:**
(Briefly describe your final architecture)


**Key Techniques Used:**
(List the main techniques: augmentation, regularization, etc.)


**Final Performance:**
- Training Accuracy:
- Validation Accuracy:
- Test Accuracy:
- Number of Parameters:

**Main Challenges and How You Addressed Them:**


**Key Takeaways:**


---

## Submission Checklist

Before submitting, verify:

- [ ] All code cells have been executed
- [ ] All analysis questions have been answered
- [ ] Normalization values in Step 1 match what you used during training
- [ ] `submission_model.pt` has been generated by the export cell
- [ ] Sanity check predictions look reasonable
- [ ] Experimentation log is filled out
- [ ] Final summary is complete

**Files to submit:**
1. This notebook (`.ipynb`)
2. `submission_model.pt`